In [2]:

import re, ast
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,
           'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,'risorse_russia':5,
           'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,
           'stabilita_russia':5,'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,
           'stabilita_rotte_cina':5,'stabilita_cina':5,'risorse_europa':5,'influenza_diplomatica_europa':5,
           'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}
ALL_T = list(DEFAULT.keys())

def eval_ternary(expr, v):
    """Eval JS ternary a?b:c in Python."""
    expr = expr.strip().rstrip(',')
    # Replace JS ternary with Python conditional
    # Pattern: cond?val1:val2
    def replace_ternary(e):
        # Iteratively replace innermost ternary
        pattern = re.compile(r'([^?:]+)\?([^?:]+):([^?:]+)')
        while '?' in e:
            e = pattern.sub(lambda m: f"({m.group(2)} if {m.group(1)} else {m.group(3)})", e, count=1)
        return e
    try:
        py_expr = replace_ternary(expr)
        return eval(py_expr, {'v': v})
    except:
        return None

def compute_delta(expr, base):
    result = eval_ternary(expr, base)
    if result is None:
        return None
    return float(result) - base

# Analisi corretta
card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
track_pos = defaultdict(int)
track_neg = defaultdict(int)

for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            expr = fn.group(1).strip().rstrip(',')
            base = DEFAULT[t]
            d = compute_delta(expr, base)
            if d is not None:
                if d > 0: track_pos[t] += 1
                elif d < 0: track_neg[t] += 1

print("=== ANALISI BIDIREZIONALITÀ CORRETTA ===")
problems = []
for t in ALL_T:
    pos = track_pos[t]; neg = track_neg[t]
    sym = "✅" if pos > 0 and neg > 0 else "❌"
    print(f"  {sym} {t}: +{pos} / -{neg}")
    if pos == 0 or neg == 0:
        problems.append((t, pos, neg))

print(f"\nTracciati problematici: {len(problems)}")
for t, pos, neg in problems:
    print(f"  {t}: pos={pos}, neg={neg}")


=== ANALISI BIDIREZIONALITÀ CORRETTA ===
  ❌ nucleare: +0 / -76
  ❌ sanzioni: +0 / -177
  ✅ opinione: +130 / -22
  ❌ defcon: +0 / -156
  ❌ risorse: +0 / -150
  ❌ stabilita: +0 / -131
  ❌ risorse_iran: +0 / -22
  ✅ forze_militari_iran: +89 / -28
  ✅ tecnologia_nucleare_iran: +11 / -25
  ✅ stabilita_iran: +38 / -49
  ✅ risorse_coalizione: +7 / -10
  ✅ influenza_diplomatica_coalizione: +36 / -46
  ✅ tecnologia_avanzata_coalizione: +17 / -16
  ✅ supporto_pubblico_coalizione: +35 / -12
  ✅ stabilita_coalizione: +7 / -50
  ✅ risorse_russia: +16 / -4
  ✅ influenza_militare_russia: +33 / -24
  ✅ veto_onu_russia: +23 / -9
  ✅ stabilita_economica_russia: +15 / -5
  ✅ stabilita_russia: +9 / -4
  ❌ risorse_cina: +0 / -32
  ✅ influenza_commerciale_cina: +55 / -14
  ✅ cyber_warfare_cina: +8 / -9
  ✅ stabilita_rotte_cina: +29 / -5
  ✅ stabilita_cina: +9 / -3
  ✅ risorse_europa: +9 / -21
  ✅ influenza_diplomatica_europa: +50 / -15
  ✅ aiuti_umanitari_europa: +5 / -3
  ✅ coesione_ue_europa: +36 / -63
 

In [5]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5}

def eval_ternary(expr, v):
    expr = expr.strip().rstrip(',')
    def replace_ternary(e):
        pattern = re.compile(r'([^?:]+)\?([^?:]+):([^?:]+)')
        while '?' in e:
            e = pattern.sub(lambda m: f"({m.group(2)} if {m.group(1)} else {m.group(3)})", e, count=1)
        return e
    try:
        py_expr = replace_ternary(expr)
        return eval(py_expr, {'v': v})
    except Exception as ex:
        return None

# Test nucleare sulle prime 10 carte
card_re = re.compile(r"card_id:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
count = 0
for m in card_re.finditer(src):
    eff_str = m.group(2)
    fn = re.search(r'nucleare\s*:\s*\(v[^)]*\)\s*=>\s*([^,}]+)', eff_str)
    if fn and count < 8:
        expr = fn.group(1).strip()
        base = DEFAULT['nucleare']
        result = eval_ternary(expr, base)
        if result is not None:
            delta = float(result) - base
            print(f"card {m.group(1)}: expr={expr!r} → result={result}, delta={delta:+.1f}")
        else:
            print(f"card {m.group(1)}: expr={expr!r} → EVAL ERROR")
        count += 1


card C025: expr='v<=7?2:v<=12?1:3' → result=1, delta=-4.0
card C026: expr='1' → result=1, delta=-4.0
card C028: expr='1' → result=1, delta=-4.0
card C030: expr='v>=11?2:1' → result=1, delta=-4.0
card C032: expr='v<=5?3:v<=10?2:1' → result=2, delta=-3.0
card C036: expr='1' → result=1, delta=-4.0
card C039: expr='1' → result=1, delta=-4.0
card C041: expr='1' → result=1, delta=-4.0


In [8]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

# I tracciati GLOBALI con effetti "assoluti" da correggere
# nucleare, sanzioni, defcon, risorse, stabilita, opinione
# Pattern: tracciato:(v)=>N dove N è un numero intero (non un'espressione con v)
# Regola: se carta Iran → nucleare sale → probabilmente era inteso come +N
# Ma dobbiamo essere precisi: cerchiamo SOLO pattern puri constant return

GLOBAL_TRACKS = ['nucleare', 'sanzioni', 'opinione', 'defcon', 'risorse', 'stabilita']

# Trova tutti i pattern: track:(v)=>COSTANTE (senza v nell'espressione)
# es: nucleare:(v)=>1, nucleare:(v)=>-1, nucleare:(v)=>2, ecc.
# Li convertiamo in: nucleare:(v)=>v+1, nucleare:(v)=>v-1, nucleare:(v)=>v+2

total_fixed = 0
new_src = src

for track in GLOBAL_TRACKS:
    # Pattern: track:(v)=>numero  (NO 'v' nell'espressione, solo costante)
    pattern = re.compile(rf'({track}\s*:\s*\(v\)\s*=>\s*)(-?\d+)([,\}}])')
    
    def replace_const(m):
        global total_fixed
        prefix = m.group(1)
        const = int(m.group(2))
        suffix = m.group(3)
        total_fixed += 1
        if const >= 0:
            return f'{prefix}v+{const}{suffix}'
        else:
            return f'{prefix}v{const}{suffix}'  # v-N
    
    new_src = pattern.sub(replace_const, new_src)

print(f"Effetti globali convertiti: {total_fixed}")

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'w') as f:
    f.write(new_src)


Effetti globali convertiti: 650


In [11]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,
           'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,'risorse_russia':5,
           'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,
           'stabilita_russia':5,'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,
           'stabilita_rotte_cina':5,'stabilita_cina':5,'risorse_europa':5,'influenza_diplomatica_europa':5,
           'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}
ALL_T = list(DEFAULT.keys())

def eval_ternary(expr, v):
    expr = expr.strip().rstrip(',')
    def replace_ternary(e):
        pattern = re.compile(r'([^?:]+)\?([^?:]+):([^?:]+)')
        while '?' in e:
            e = pattern.sub(lambda m: f"({m.group(2)} if {m.group(1)} else {m.group(3)})", e, count=1)
        return e
    try:
        return eval(replace_ternary(expr), {'v': v})
    except:
        return None

def compute_delta(expr, base):
    r = eval_ternary(expr, base)
    if r is None: return None
    return float(r) - base

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
track_pos = defaultdict(int)
track_neg = defaultdict(int)

for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            d = compute_delta(fn.group(1), DEFAULT[t])
            if d is not None:
                if d > 0: track_pos[t] += 1
                elif d < 0: track_neg[t] += 1

print("=== POST FIX GLOBALI ===")
problems = []
for t in ALL_T:
    pos = track_pos[t]; neg = track_neg[t]
    sym = "✅" if pos > 0 and neg > 0 else "❌"
    print(f"  {sym} {t}: +{pos} / -{neg}")
    if pos == 0 or neg == 0:
        problems.append((t, pos, neg))

print(f"\nProblemi residui: {len(problems)}")


=== POST FIX GLOBALI ===
  ✅ nucleare: +28 / -48
  ✅ sanzioni: +80 / -97
  ✅ opinione: +130 / -22
  ✅ defcon: +54 / -102
  ✅ risorse: +90 / -60
  ✅ stabilita: +70 / -61
  ❌ risorse_iran: +0 / -22
  ✅ forze_militari_iran: +89 / -28
  ✅ tecnologia_nucleare_iran: +11 / -25
  ✅ stabilita_iran: +38 / -49
  ✅ risorse_coalizione: +7 / -10
  ✅ influenza_diplomatica_coalizione: +36 / -46
  ✅ tecnologia_avanzata_coalizione: +17 / -16
  ✅ supporto_pubblico_coalizione: +35 / -12
  ✅ stabilita_coalizione: +7 / -50
  ✅ risorse_russia: +16 / -4
  ✅ influenza_militare_russia: +33 / -24
  ✅ veto_onu_russia: +23 / -9
  ✅ stabilita_economica_russia: +15 / -5
  ✅ stabilita_russia: +9 / -4
  ❌ risorse_cina: +0 / -32
  ✅ influenza_commerciale_cina: +55 / -14
  ✅ cyber_warfare_cina: +8 / -9
  ✅ stabilita_rotte_cina: +29 / -5
  ✅ stabilita_cina: +9 / -3
  ✅ risorse_europa: +9 / -21
  ✅ influenza_diplomatica_europa: +50 / -15
  ✅ aiuti_umanitari_europa: +5 / -3
  ✅ coesione_ue_europa: +36 / -63
  ✅ stabilita_e

In [14]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

# risorse_iran: tutte le carte hanno v-1 (diminuzione), nessuna ha v+1
# Dobbiamo trovare carte Iran Economico e aggiungere risorse_iran:(v)=>v+1
# risorse_cina: tutte hanno v-1, nessuna v+1
# Dobbiamo trovare carte Cina Economico e aggiungere risorse_cina:(v)=>v+1

card_re = re.compile(
    r"\{\s*card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)',\s*op_points:(\d+),\s*deck_type:'([^']+)',\s*description:'([^']*)'",
    re.DOTALL
)
matches = list(card_re.finditer(src))

fixes = [
    ('risorse_iran', 'Iran', 'Economico', 'v+1', 8),
    ('risorse_cina', 'Cina', 'Economico', 'v+1', 8),
]

new_src = src
total = 0

for track, target_faz, target_type, delta_expr, max_cards in fixes:
    count = 0
    for idx, m in enumerate(matches):
        if count >= max_cards:
            break
        cid, cname, faz, ctype, op, deck, desc = m.groups()
        if faz != target_faz or ctype != target_type:
            continue
        
        card_search = f"card_id:'{cid}'"
        pos = new_src.find(card_search)
        if pos == -1:
            continue
        
        eff_m = re.search(r'(effects:\{)([^}]+)(\})', new_src[pos:pos+800])
        if not eff_m:
            continue
        
        eff_str = eff_m.group(2)
        if track + ':' in eff_str:
            continue
        
        new_eff = eff_m.group(1) + eff_m.group(2) + f', {track}:(v)=>{delta_expr}' + eff_m.group(3)
        abs_pos = pos + eff_m.start()
        new_src = new_src[:abs_pos] + new_eff + new_src[abs_pos + len(eff_m.group(0)):]
        count += 1
        total += 1

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'w') as f:
    f.write(new_src)

print(f"Effetti aggiunti: {total}")


Effetti aggiunti: 0
